In [1]:
!pip install -q transformers datasets sacrebleu sentencepiece accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.0 MB/s eta 0:00:00


In [2]:
import os, json, math, time
from datasets import load_dataset#, load_metric
from transformers import (
    MBart50TokenizerFast,
    MBartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)
import torch
import numpy as np

In [ ]:
# replace the old load_cell
raw = load_dataset("wmt17", "zh-en",
                   split={"train":"train[:1%]",
                          "validation":"validation",
                          "test":"test"})
src, tgt = "en", "zh"
def flip(batch):
    batch["translation"] = {"en": batch["translation"]["en"],
                            "zh": batch["translation"]["zh"]}
    return batch
raw = raw.map(flip)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

zh-en/train-00000-of-00013.parquet:   0%|          | 0.00/286M [00:00<?, ?B/s]

zh-en/train-00001-of-00013.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

zh-en/train-00002-of-00013.parquet:   0%|          | 0.00/281M [00:00<?, ?B/s]

zh-en/train-00003-of-00013.parquet:   0%|          | 0.00/278M [00:00<?, ?B/s]

zh-en/train-00004-of-00013.parquet:   0%|          | 0.00/277M [00:00<?, ?B/s]

zh-en/train-00005-of-00013.parquet:   0%|          | 0.00/281M [00:00<?, ?B/s]

In [ ]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tok = MBart50TokenizerFast.from_pretrained(model_name, src_lang="en_XX", tgt_lang="zh_CN")
model = MBartForConditionalGeneration.from_pretrained(model_name)


In [ ]:
max_src, max_tgt = 128, 128
def encode(ex):
    en_sent = [item["en"] for item in ex["translation"]]
    zh_sent = [item["zh"] for item in ex["translation"]]
    # 一次调用同时编码源端和目标端
    model_inputs = tok(
        en_sent,
        text_target=zh_sent,
        max_length=max_src,
        truncation=True,
        padding=False,          # 动态 padding，由 data_collator 完成
    )
    return model_inputs

tokenised = raw.map(encode, batched=True, remove_columns=raw["train"].column_names)

In [ ]:
data_coll = DataCollatorForSeq2Seq(tok, model=model)

args = Seq2SeqTrainingArguments(
    output_dir="mbart-en-zh-wmt14",
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=500,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=max_tgt,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

In [ ]:
!pip install -q evaluate

In [ ]:
import evaluate

bleu = evaluate.load("sacrebleu")
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tok.pad_token_id)
    pred_str = tok.batch_decode(preds, skip_special_tokens=True)
    label_str = tok.batch_decode(labels, skip_special_tokens=True)
    bleu_score = bleu.compute(predictions=pred_str, references=[[r] for r in label_str])["score"]
    return {"bleu": bleu_score}

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenised["train"],
    eval_dataset=tokenised["validation"],
    processing_class=tok,
    data_collator=data_coll,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.args

In [ ]:
trainer.train()


In [ ]:
trainer.evaluate(tokenised["test"])

In [ ]:
def translate_en_zh(sentence: str) -> str:
    model.eval()
    with torch.no_grad():
        inputs = tok(sentence, return_tensors="pt").to(model.device)
        generated = model.generate(**inputs,
                                   forced_bos_token_id=tok.lang_code_to_id["zh_CN"],
                                   max_length=150,
                                   num_beams=5,
                                   early_stopping=True)
    return tok.batch_decode(generated, skip_special_tokens=True)[0]

translate_en_zh("Machine translation is not quite solved yet.")

In [ ]:
trainer.save_model("mbart-en-zh-wmt14-best")
tok.save_pretrained("mbart-en-zh-wmt14-best")